<a href="https://colab.research.google.com/github/xc308/Dataset_preparation_Fine_Tuning/blob/main/Fine_tune_Model_with_bfloat16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**:

-  train a large model on a single GPU using the bfloat16 data type

- load the model parameters using the bfloat16 data type.
- Combining this with LoRA, can fine-tune a 4B model on the GPU available.



**Steps**

- loading of parameters in bfloat16 in Keras
- explain how bfloat16 significantly reduces the memory footprint of a model
- Fine-tune a large model on a resource-constrained GPU

In [1]:
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For setting system variables.

from google.colab import userdata # For using Colab secrets.

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

os.environ["KERAS_BACKEND"] = "jax"  # Set the Keras backend to JAX.

# Disable the command buffer pre-allocation to free up memory.
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_command_buffer="
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import keras # For defining and training models.
import keras_hub # For loading the Keras implementation of Gemma.
import pandas as pd # For loading the dataset.
from textwrap import fill # For formatting long paragraphs.
from ai_foundations import formatting # For formatting the training data.

keras.utils.set_random_seed(812)  # For Keras layers.

  Cloning https://github.com/google-deepmind/ai-foundations.git (to revision main) to /tmp/pip-req-build-hq2lujh0
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/ai-foundations.git /tmp/pip-req-build-hq2lujh0
  Resolved https://github.com/google-deepmind/ai-foundations.git to commit 524d6114bbce631dafc00ba3496607a0bc60c804
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


**Load and process data**

- defines a function format_question that formats a prompt

- loads the Africa Galore QA dataset

- and processes the individual questions so that the data can be used to fine-tune a model to generate answers

In [2]:
def format_question(
        question: str,
        sot = "<start_of_turn>",
        eot = "<end_of_turn>"
)-> str:

    formatted_q = f"{sot}user\n{question}{eot}\n"

    return formatted_q


In [3]:
# Load the question-answer dataset.
africa_galore_qa = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_qa_v2.json"
)


In [4]:
africa_galore_qa.head(5)

,category,name,question,answer
0,Textile,Kente Cloth,What is Kente Cloth?,The vibrant colors and intricate patterns of K...
1,Textile,Bogolanfini (Mud Cloth),What is Bogolanfini (Mud Cloth)?,"Bogolanfini, or mud cloth, from Mali, is a tex..."
2,Textile,Adire,What is Adire?,"Adire, a resist-dyed indigo cloth from Nigeria..."
3,Textile,Kanga,What is Kanga?,"Kanga, a colorful printed cloth from East Afri..."
4,Textile,Ankara,What is Ankara?,"Ankara, also known as African wax print fabric..."


In [5]:
questions = []  # List of formatted questions.
answers = []  # List of formatted answers.

for idx, row in africa_galore_qa.iterrows():
    # Run the format_qa function from the previous lab to format the question
    # and the answer.
    question, answer = formatting.format_qa(row)
    questions.append(question)
    answers.append(answer)

# Show the first set of input and output.
print(questions[0])
print(fill(answers[0], replace_whitespace=False))

<start_of_turn>user
What is Kente Cloth?<end_of_turn>

<start_of_turn>model
Category: Textile
The vibrant colors and
intricate patterns of Kente cloth, a symbol of Ghanaian royalty and
prestige, tell stories of history, culture, and social status. Woven
on narrow looms by skilled artisans, each strip of Kente is a
testament to patience and artistry. The geometric designs, rich with
symbolism, represent proverbs, historical events, and important
figures. Worn during special occasions and ceremonies, Kente cloth
embodies the spirit of Ghana, its vibrant culture, and its rich
history. From the bright yellows and golds representing royalty to the
deep blues and greens symbolizing spirituality, Kente is a visual
language, a wearable expression of Ghanaian identity and
heritage.<end_of_turn>


In [6]:
questions = []
answers = []

for idx, row in africa_galore_qa.iterrows():
    question, answer = formatting.format_qa(row)
    questions.append(question)
    answers.append(answer)

print(questions[0])
print(fill(answers[0], replace_whitespace=False))

<start_of_turn>user
What is Kente Cloth?<end_of_turn>

<start_of_turn>model
Category: Textile
The vibrant colors and
intricate patterns of Kente cloth, a symbol of Ghanaian royalty and
prestige, tell stories of history, culture, and social status. Woven
on narrow looms by skilled artisans, each strip of Kente is a
testament to patience and artistry. The geometric designs, rich with
symbolism, represent proverbs, historical events, and important
figures. Worn during special occasions and ceremonies, Kente cloth
embodies the spirit of Ghana, its vibrant culture, and its rich
history. From the bright yellows and golds representing royalty to the
deep blues and greens symbolizing spirituality, Kente is a visual
language, a wearable expression of Ghanaian identity and
heritage.<end_of_turn>


In [7]:
# Prepare the data dictionary for fine-tuning Gemma.
data = {
    "prompts": questions,
    "responses": answers
}

**Load Gemma-4B parameters**

- if load Gemma-4B with parameters stored in 32-bit default format, it would be too large to fit in the GPU's memory

- The solution is to load the model using a more memory-efficient number format: bfloat16

In [1]:
model = keras_hub.models.Gemma3CausalLM.from_preset(preset="gemma3_4b_text", dtype="bfloat16")
model.summary()

NameError: name 'keras_hub' is not defined

**Inspect some of the model weights to verify that its parameters are represented as bfloat16**

In [ ]:
# Access the first transformer block.
first_transformer_block = model.backbone.get_layer("decoder_block_0")

# Access the attention layer.
attention_layer = first_transformer_block.attention

# Get the weight matrix for the query projections, and check its dtype.
query_weights = attention_layer.query_dense.kernel

print(f"The model's weight precision is {query_weights.dtype}.")

In [8]:
first_transformer_block = model.backbone.get_layer("decoder_block_0")

attention_layer = first_transformer_block.attention

query_weights = attention_layer.query_dense.kernel # weight matrix for the query projections

print(f"The model's weight precision is {query_weights.dtype}.")

The model's weight precision is bfloat16.


**Activate LoRA**

- Even with bfloat16 dtype, we still cannot perform full-parameter fine-tuning, but LoRA

- Thelow-rank matrices constitute a small fraction of the total parameters.
    - This means that the GPU memory needs to store gradients, activations, and optimizer states for only a small number of parameters.

- drastically reduces the overall memory requirement.
    - The lower memory requirement is why you can fine-tune a 4B model on a T4 GPU.

In [9]:
model.backbone.enable_lora(rank=4)
model.summary()

Preprocessor: "gemma3_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma3_tokenizer (Gemma3Tokenizer)                            │                      Vocab size: 262,144 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma3_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma3_backbone               │ (None, None, 2560)        │   3,884,346,880 │ padding_mask[0][0],        │
│ (Gemma3Backbone)              │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 262144)      │     671,088,640 │ gemma3_backbone[0][0]      │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 3,884,346,880 (7.24 GB)

 Trainable params: 4,247,552 (8.10 MB)

 Non-trainable params: 3,880,099,328 (7.23 GB)

**Compare the trainable parameters between Base model and LoRA enabled**

- Gemma3 4B: 3,884,346,880 (7.24 GB)

- LoRA enabled in the backbone: 4,247,552 (8.10 MB)

## Fine-tune Gemma-4B with bfloat16 using LoRA

- After LoRA is enabled, need to configure the training process.
    - This involves setting the hyperparameters that will guide the learning

    - Training with a low-precision format e.g., bfloat16, requires more careful hyperparameter selection than standard 32-bit training

    - As the numbers are less precise, the learning process can be more sensitive, so important to use settings that result in a stable training process

- The following AdamW optimizer with a set of values that have been found to work well for stable bfloat16 fine-tuning.

In [ ]:
# Set hyperparameters.
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
    beta_1=0.9,
    beta_2=0.95,
    epsilon=1e-6,
    clipnorm=0.5,
)

# Determine the number of epochs.
num_epochs = 4

# Set the maximum length.
model.preprocessor.sequence_length = 400

# Compile the optimizer.
model.compile(
    optimizer=optimizer,
)

## Monitor training progress##

- need a method to evaluate the training progress beyond the loss:

    - by sampling generations from the model after each epoch


     - a helper function that will automatically run code at the end of each epoch.
     - In this case, it will generate and print outputs for three specific test prompts. This allows you to monitor the model's progress in real-time.   

- Three different questions test differet facets:

    - "What is Kente cloth?": This tests if the model is learning the specific knowledge from the fine-tuning dataset.

    - "What is Kilimanjaro?": This tests generalization, as "Mount Kilimanjaro" features in the fine-tuning dataset but the word "Mount" is omitted here to make it more difficult.

    - "What is Tokyo?": Since there is nothing about Tokyo or Japan in the fine-tuning dataset, this checks if the model retains its pre-trained knowledge while learning the new task.



In [ ]:
# Define a list of prompts you want to check after each epoch.
test_prompts = [
    "What is Kente cloth?",
    "What is Kilimanjaro?",
    "What is Tokyo?"
]

class GenerationMonitor(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print(f"\n--- Generations after epoch {epoch + 1} ---")
        for prompt in test_prompts:
            # Format the prompt correctly for the model.
            formatted_prompt = format_question(prompt)

            # Generate and print the output.
            output = self.model.generate(formatted_prompt, max_length=150)
            print(output)
            print("-" * 20)

## fine-tune the model using LoRA and the bfloat16##


In [ ]:
# Create an instance of the monitoring callback.
generation_callback = GenerationMonitor()

# Train the model.
history = model.fit(
    data,
    epochs=num_epochs,
    batch_size=1,
    callbacks=[generation_callback],
)

**loading the model with bfloat16 instead of 32-bit floating point numbers - can not only load the model but also fine-tune it with LoRA.**

-